# MindSync Stage 1 — Text Training (RoBERTa-Large on GoEmotions)

**Setup (one-time):**
1. Runtime → Change runtime type → **T4 GPU**
2. Tools → 🔑 **Secret** → add a secret named `HF_TOKEN` with your Hugging Face write token. Toggle the side switch ON for this notebook.
3. Runtime → **Run All**

**What it does:**
- Downloads GoEmotions from Hugging Face (auto)
- Fine-tunes RoBERTa-Large for 4-cluster emotion classification (5 epochs)
- Saves the best checkpoint per epoch to Google Drive (resume-safe)
- Uploads the best checkpoint to `Ubaida1/mindsync-text-model` on completion

**Time:** ~2-3 hrs on T4. Keep the tab open.

## 0 · Mount Google Drive + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
CKPT_DIR = pathlib.Path('/content/drive/MyDrive/mindsync')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint dir:', CKPT_DIR)


In [ ]:
# Pin transformers to a working version (RobertaTokenizerFast was broken in 5.x)
!pip install -q 'transformers==4.45.2' 'datasets>=2.18' 'scikit-learn' 'huggingface_hub>=0.20'
import transformers; print('transformers:', transformers.__version__)


## 1 · HF token (from Colab Secret `HF_TOKEN`)

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN secret in Tools → Secret'
from huggingface_hub import login
login(token=HF_TOKEN)
print('Logged in to Hugging Face.')


## 2 · Cluster mapping (27 GoEmotions → 4 clinical clusters)

In [ ]:
EMOTION_TO_CLUSTER = {
    'grief': 'Distress', 'nervousness': 'Distress', 'fear': 'Distress',
    'sadness': 'Distress', 'remorse': 'Distress',
    'joy': 'Resilience', 'admiration': 'Resilience', 'excitement': 'Resilience',
    'relief': 'Resilience', 'amusement': 'Resilience', 'approval': 'Resilience',
    'curiosity': 'Resilience', 'desire': 'Resilience', 'gratitude': 'Resilience',
    'love': 'Resilience', 'optimism': 'Resilience', 'pride': 'Resilience',
    'realization': 'Resilience',
    'anger': 'Aggression', 'annoyance': 'Aggression', 'disgust': 'Aggression',
    'disapproval': 'Aggression', 'embarrassment': 'Aggression',
    'confusion': 'Ambiguity', 'disappointment': 'Ambiguity', 'surprise': 'Ambiguity',
    'caring': 'Ambiguity', 'neutral': 'Ambiguity',
}
CLUSTERS = ['Distress', 'Resilience', 'Aggression', 'Ambiguity']
CLUSTER_TO_IDX = {c: i for i, c in enumerate(CLUSTERS)}


## 3 · Load + preprocess GoEmotions

In [ ]:
from datasets import load_dataset
ds = load_dataset('google-research-datasets/go_emotions', 'simplified')
label_names = ds['train'].features['labels'].feature.names
print('Splits:', {k: len(v) for k, v in ds.items()})
print('Labels:', len(label_names))


In [ ]:
from transformers import AutoTokenizer
MODEL_NAME = 'roberta-large'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 128

def map_to_cluster_idx(label_ids):
    if not label_ids: return None
    return CLUSTER_TO_IDX[EMOTION_TO_CLUSTER.get(label_names[label_ids[0]], 'Ambiguity')]

def preprocess(batch):
    enc = tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LEN)
    enc['labels'] = [map_to_cluster_idx(l) for l in batch['labels']]
    return enc

ds_proc = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)
ds_proc = ds_proc.filter(lambda x: x['labels'] is not None)
ds_proc.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
print({k: len(v) for k, v in ds_proc.items()})


## 4 · MindSyncTextModel (RoBERTa-Large + 4-class head)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import RobertaModel

class MindSyncTextModel(nn.Module):
    HIDDEN_SIZE = 1024
    def __init__(self, model_name='roberta-large', num_classes=4, dropout=0.1):
        super().__init__()
        self.encoder = nn.Module(); self.encoder.encoder = RobertaModel.from_pretrained(model_name)
        self.encoder.hidden_size = self.encoder.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Module()
        h = self.encoder.hidden_size
        self.classifier.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(h, h // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(h // 2, num_classes),
        )
    def forward(self, input_ids, attention_mask):
        out = self.encoder.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        logits = self.classifier.classifier(cls)
        return {'embedding': cls, 'logits': logits}


## 5 · Training loop with per-epoch checkpoint to Drive

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from collections import Counter

device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))

BATCH = 16
EPOCHS = 5
LR = 2e-5

# weighted sampler for class balance
train_labels = ds_proc['train']['labels'].tolist()
counts = Counter(train_labels)
weights = [1.0 / counts[l] for l in train_labels]
sampler = WeightedRandomSampler(weights, num_samples=len(train_labels), replacement=True)
train_loader = DataLoader(ds_proc['train'], batch_size=BATCH, sampler=sampler)
val_loader = DataLoader(ds_proc['validation'], batch_size=32)
test_loader = DataLoader(ds_proc['test'], batch_size=32)

model = MindSyncTextModel(num_classes=4).to(device)
optim = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
sched = get_linear_schedule_with_warmup(optim, int(0.1 * total_steps), total_steps)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, gold = [], []
    for batch in loader:
        ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
        out = model(ids, mask)
        preds.extend(out['logits'].argmax(-1).cpu().tolist())
        gold.extend(batch['labels'].tolist())
    return accuracy_score(gold, preds), f1_score(gold, preds, average='macro')

best_f1 = 0.0
for ep in range(1, EPOCHS + 1):
    model.train(); total = 0.0; n = 0
    for batch in tqdm(train_loader, desc=f'Epoch {ep}'):
        ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
        y = batch['labels'].to(device)
        optim.zero_grad()
        out = model(ids, mask)
        loss = F.cross_entropy(out['logits'], y)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step(); sched.step()
        total += loss.item() * y.size(0); n += y.size(0)
    val_acc, val_f1 = evaluate(val_loader)
    print(f'Epoch {ep} | loss {total/n:.4f} | val acc {val_acc:.4f} | val F1 {val_f1:.4f}')
    # save per-epoch checkpoint to Drive
    torch.save(model.state_dict(), CKPT_DIR / f'text_epoch{ep}.pt')
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), CKPT_DIR / 'best_text_model.pt')
        print(f'  ✓ Saved best (F1={best_f1:.4f})')

test_acc, test_f1 = evaluate(test_loader)
print(f'\nFINAL TEST: acc {test_acc:.4f} | F1 {test_f1:.4f}')


## 6 · Upload best checkpoint to HF Model repo

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
REPO = 'Ubaida1/mindsync-text-model'
api.create_repo(repo_id=REPO, repo_type='model', exist_ok=True)
api.upload_file(
    path_or_fileobj=str(CKPT_DIR / 'best_text_model.pt'),
    path_in_repo='best_text_model.pt',
    repo_id=REPO, repo_type='model',
    commit_message=f'Stage 1: RoBERTa-Large, test acc {test_acc:.4f}, F1 {test_f1:.4f}',
)
print('Uploaded → https://huggingface.co/' + REPO)


## Done!
Now tell Claude **"Stage 1 done"** with the printed `test acc` / `F1`. He'll:
1. Restart the HF Space so it picks up the new checkpoint
2. Verify predictions on the cloud backend
3. Prepare **Stage 2: Audio training**
